# Práctica 1D: Información mutua

## Realizado por: Eduardo Alanís García

# Instrucciones

1. Preprocesa el texto y conviértelo a minúsculas, quita acentos a las vocales y los siguientes caracteres:

    ```
    ;:,.\-"'/()[]¿?¡!{}~<>|«»-—'
    ```

    Reemplaza las tabulaciones y saltos de línea con un espacio.

    Elimina las stopwords.

2. Encuentra las asociaciones de palabras más significativas usando la medida de información mutua para cada conjunto de datos. Cada conjunto de datos se procesa por separado.

    Los archivos son:

    - archivo_emojis_Proceso.csv
    - archivo_emojis_Elpais.csv
    - archivo_emojis_Elfinanciero.csv

3. Usa el contenido de la columna `title`. Los puedes leer con:

    ```
    data = pd.read_csv("<ruta>/archivo_emojis_Elfinanciero.csv")
    text = data['title'].to_list()
    text = " ".join(text)
    ```

4. Crea un notebook que calcule la información mutua de un texto. Usa una ventana de dos palabras para calcular las probabilidades conjuntas $p(x, y)$.

5. Incluye las asociaciones más importantes respecto a la medida de información mutua para cada conjunto de datos proporcionado. Es decir, se deben de calcular todas las asociaciones entre el vocabulario único. Muestra las $10$ asociaciones más importantes que aparezcan al menos $10$ veces en el texto e indica su índice de información mutua.

### Cargar google drive

In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Verificar existencia de los archivos

In [42]:
import os

direccion = "/content/drive/MyDrive/MCDI-INFOTEC/1. Primer semestre/Manejo y procesamiento de la informacion/Unidad 1/Práctica 1D Información mutua/"

archivos = ["archivo_emojis_Proceso.csv", "archivo_emojis_Elpais.csv", "archivo_emojis_Elfinanciero.csv"]

for archivo in archivos:
  ruta = os.path.join(direccion, archivo)
  print(archivo, "Existe" if os.path.exists(ruta) else "No Existe")

archivo_emojis_Proceso.csv Existe
archivo_emojis_Elpais.csv Existe
archivo_emojis_Elfinanciero.csv Existe


### Función leer archivos

In [43]:
import pandas as pd

def leer_archivo(nombre_archivo, direccion=direccion):
  ruta = os.path.join(direccion, nombre_archivo)

  data = pd.read_csv(ruta)
  texto = data['title'].to_list()
  texto = " ".join(str(t) for t in texto)

  return texto

### Función de quitar acentos

In [44]:
import unicodedata

def remove_accents(input_str):
  nfkd_form = unicodedata.normalize('NFKD', input_str)
  return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

### Caracteres a eliminar

In [45]:
caracteres_eliminar = ";:,.\\-\"'/()[]¿?¡!{}~<>|«»-—’"

### Función preprocesar texto

In [46]:
def preprocesar_texto(texto):
  texto = texto.lower()  # Minusculas
  texto = remove_accents(texto)  # Acentos
  texto = texto.replace('\t', ' ').replace('\n', ' ')  # Tabulaciones y saltos de linea

  for c in caracteres_eliminar:
    texto = texto.replace(c, '')  # Eliminar caracteres mencionados

  return texto

### Función para quitar stopwords

In [47]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
_STOPWORDS = stopwords.words("spanish")

def remove_stopwords(text):
  new_text = []
  for word in text.split():
    if word in _STOPWORDS:
      continue
    else:
      new_text.append(word)
  return " ".join(new_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Función cálculo de frecuencias de palabras

In [48]:
from collections import Counter

def calcular_frecuencias_palabras(texto):
  palabras = texto.split()
  frecuencias = Counter(palabras)
  ordenado = frecuencias.most_common()

  palabras_out = []
  valores = []

  for palabra, frecuencia in ordenado:
    palabras_out.append(palabra)
    valores.append(frecuencia)

  guardar = {"palabras": palabras_out, "valores": valores}

  return guardar

### Función cálculo de frecuencias de bigramas

In [49]:
def calcular_frecuencias_bigramas(texto):
  palabras = texto.split()
  bigramas = [f"{palabras[i]} {palabras[i+1]}" for i in range(len(palabras) - 1)]

  frecuencias = Counter(bigramas)
  ordenado = frecuencias.most_common()

  bigramas_out = []
  valores = []

  for bigrama, frecuencia in ordenado:
    bigramas_out.append(bigrama)
    valores.append(frecuencia)

  guardar = {"bigramas": bigramas_out, "valores": valores}

  return guardar

## Información mutua

La información mutua $I(x_i, y_i)$, es una medida de la cantidad de información que una variable aleatoria contiene de otra variable aleatoria, es decir, es la reducción de la incertidumbre de una variable aleatoria debido al conocimiento de la otra variable.

$$ I(x_i, y_i) = \log_2\frac{p(x_i, y_i)}{p(x_i)\,p(y_i)} $$

donde $p(x_i,y_i)$ es la frecuencia relativa del bigrama $(x_i, y_i)$ respecto al total de bigramas, y $p(x_i)$, $p(y_i)$ se pueden obtener de las frecuencias relativas de los datos.

La frecuencia relativa $n_i$ es el cociente entre la frecuencia absoluta $f_i$
 y el número total de datos $N$

 $$p(x_i) = n_i = \frac{f_i}{N}$$

 La información mutua se ha aplicado en el descubrimiento de las asociaciones de palabras que para las personas son de sentido común, pero para una computadora no lo son.

In [50]:
import math

def calcular_informacion_mutua(texto):
  palabras = texto.split()
  N = len(palabras)  # total de unigramas

  frecuencias_unigramas = Counter(palabras)

  bigramas = [(palabras[i], palabras[i + 1]) for i in range(len(palabras) - 1)]
  frecuencias_bigramas = Counter(bigramas)
  N_bigramas = len(bigramas)  # total de bigramas

  resultados = {}

  for (x, y), f_xy in frecuencias_bigramas.items():
    p_xy = f_xy / N_bigramas
    p_x = frecuencias_unigramas[x] / N
    p_y = frecuencias_unigramas[y] / N

    im = math.log2(p_xy / (p_x * p_y))

    resultados[f"{x} {y}"] = {"im": im, "frecuencia": f_xy}



  return resultados

### Función para obtener las asociaciones más importantes

In [51]:
def top_asociaciones(informacion_mutua, top_n=10, frecuencia_minima=10):
  filtradas = []

  for bigrama, valores in informacion_mutua.items():
    if valores["frecuencia"] >= frecuencia_minima:
      filtradas.append((bigrama, valores))

  filtradas.sort(key=lambda x: x[1]["im"], reverse=True)

  return filtradas[:top_n]

### Generar los resultados para cada conjunto de datos

In [52]:
resultados = {}

for archivo in archivos:
  texto_original = leer_archivo(nombre_archivo=archivo)
  texto_procesado = preprocesar_texto(texto=texto_original)
  texto_sin_stopwords = remove_stopwords(texto_procesado)

  frecuencias_unigramas = calcular_frecuencias_palabras(texto_sin_stopwords)
  frecuencias_bigramas = calcular_frecuencias_bigramas(texto_sin_stopwords)
  informacion_mutua = calcular_informacion_mutua(texto_sin_stopwords)
  top10 = top_asociaciones(informacion_mutua, top_n=10, frecuencia_minima=10)

  resultados[archivo] = {
      "texto_procesado": texto_sin_stopwords,
      "frecuencias_unigramas": frecuencias_unigramas,
      "frecuencias_bigramas": frecuencias_bigramas,
      "informacion_mutua": informacion_mutua,
      "top10": top10
  }

In [53]:
for archivo in archivos:
  print(f"{archivo}")
  top10 = resultados[archivo]["top10"]

  datos_top10 = []

  for bigrama, valores in top10:
      datos_top10.append(
          (bigrama, valores["frecuencia"], valores["im"])
      )


  df_top10 = pd.DataFrame(datos_top10, columns=["asociacion", "frecuencia","informacion_mutua"])

  display(df_top10)
  print()

archivo_emojis_Proceso.csv


,asociacion,frecuencia,informacion_mutua
0,salinas pliego,10,11.104254
1,elena poniatowska,11,11.019365
2,delfina gomez,11,11.019365
3,conferencia mananera,11,10.812915
4,tren maya,13,10.768168
5,adan augusto,19,10.745800
6,cambio climatico,10,10.544166
7,alejandra moral,14,10.457231
8,moreno valle,11,10.450344
9,movimiento ciudadano,12,10.395303



archivo_emojis_Elpais.csv


,asociacion,frecuencia,informacion_mutua
0,lucrecia medici,10,14.685012
1,dorados llamativos,10,14.685012
2,llamativos bisuteria,10,14.685012
3,svenja flasspohler,10,14.685012
4,‘te deseo”,10,14.685012
5,gypsy rose,10,14.685012
6,rose blanchard,10,14.685012
7,brad pitt,10,14.547508
8,nagorno karabaj,11,14.547508
9,skid row,10,14.547508



archivo_emojis_Elfinanciero.csv


,asociacion,frecuencia,informacion_mutua
0,rutilio escandon,10,14.309540
1,sinead oconnor,10,14.309540
2,rosita pelayo,10,14.309540
3,britney spears,10,14.172037
4,salomon jara,10,14.034533
5,natanael cano,10,13.931029
6,cristiano ronaldo,12,13.931029
7,galilea montijo,13,13.931029
8,hipolito mora,11,13.931029
9,octavio ocana,10,13.909002
